In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("patient_health_records_1000_rows.csv")
df.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P100001,69.0,Male,North,19.7,109.5,165.5,103.0,0
1,P100002,32.0,Female,South,19.5,119.9,186.0,78.3,0
2,P100003,78.0,Female,West,14.2,112.8,112.2,91.9,0
3,P100004,38.0,Male,East,20.5,97.4,219.9,114.9,0
4,P100005,41.0,Male,South,26.1,107.1,238.0,81.1,0


# Part - A : Handling missing values

## 1. Identify missing values and provide a summary report(percentage per column)

In [4]:
missing_data = df.isnull().sum()
print(f"Missing values per column:\n{missing_data}")

missing_percentage = (missing_data / len(df)) * 100
print(f"\nPercentage of missing values per column:\n{missing_percentage}")


Missing values per column:
patient_id         0
age               30
gender            20
region            20
bmi               40
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
dtype: int64

Percentage of missing values per column:
patient_id        0.0
age               3.0
gender            2.0
region            2.0
bmi               4.0
blood_pressure    0.0
cholesterol       3.0
glucose           3.0
disease_risk      0.0
dtype: float64


## 2. Apply the following imputation techniques and compare results:


### - Simple Imputer(Numerical):


In [10]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
df["bmi"] = imputer.fit_transform(df[["bmi"]])
print("\nMissing values after imputation:\n", df.isnull().sum())



Missing values after imputation:
 patient_id         0
age               30
gender            20
region             0
bmi                0
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
dtype: int64


### - Simple Imputer(Categorical): 

In [12]:
imputer = SimpleImputer(strategy="most_frequent")
df[["region"]] = imputer.fit_transform(df[["region"]])
print("\nMissing values after imputation:\n", df.isnull().sum())


Missing values after imputation:
 patient_id         0
age               30
gender             0
region             0
bmi                0
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
dtype: int64


### - Most Frequent Imputation:

In [13]:
imputer = SimpleImputer(strategy="most_frequent")
df[["gender"]] = imputer.fit_transform(df[["gender"]])
print("\nMissing values after imputation:\n", df.isnull().sum())


Missing values after imputation:
 patient_id         0
age               30
gender             0
region             0
bmi                0
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
dtype: int64


### - Missing Indicator + Random Sample Imputation:

In [16]:
df["BMI_Missing"] = df["bmi"].isnull().astype(int)

random_values = df["bmi"].dropna().sample(
    df["bmi"].isnull().sum(),
    replace=True,
    random_state=10
)

df.loc[df["bmi"].isnull(), "bmi"] = random_values.values

print(df[["bmi", "BMI_Missing"]].head())

    bmi  BMI_Missing
0  19.7            0
1  19.5            0
2  14.2            0
3  20.5            0
4  26.1            0


### - KNN Imputer:

In [17]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
df[["bmi"]] = imputer.fit_transform(df[["bmi"]])
print("\nMissing values after KNN imputation:\n", df.isnull().sum())



Missing values after KNN imputation:
 patient_id         0
age               30
gender             0
region             0
bmi                0
blood_pressure     0
cholesterol       30
glucose           30
disease_risk       0
BMI_Missing        0
dtype: int64


### - MICE Algorithm:

In [18]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

numeric_data = df.select_dtypes(include=["number"])

mice = IterativeImputer(random_state=42)

df[numeric_data.columns] = mice.fit_transform(numeric_data)

print(df.head())

  patient_id   age  gender region   bmi  blood_pressure  cholesterol  glucose  \
0    P100001  69.0    Male  North  19.7           109.5        165.5    103.0   
1    P100002  32.0  Female  South  19.5           119.9        186.0     78.3   
2    P100003  78.0  Female   West  14.2           112.8        112.2     91.9   
3    P100004  38.0    Male   East  20.5            97.4        219.9    114.9   
4    P100005  41.0    Male  South  26.1           107.1        238.0     81.1   

   disease_risk  BMI_Missing  
0           0.0          0.0  
1           0.0          0.0  
2           0.0          0.0  
3           0.0          0.0  
4           0.0          0.0  


# Part - B : Handling Outliers

## 3. Detect and remove outliers using:

### - Z-score method:

In [20]:
z_scores = np.abs((df['cholesterol'] - df['cholesterol'].mean()) / df['cholesterol'].std())
outliers = df[z_scores > 3]
print(outliers)

    patient_id   age  gender region   bmi  blood_pressure  cholesterol  \
78     P100079  28.0  Female  North  30.9           118.4        392.2   
97     P100098  64.0    Male  North  29.0           116.7        330.4   
118    P100119  45.0    Male  South  25.4           125.4        390.0   
197    P100198  65.0  Female   West  23.1           101.5        410.3   
221    P100222  50.0    Male   East  20.2           101.2        413.9   
380    P100381  67.0    Male   West  29.8            92.6        393.7   
537    P100538  50.0    Male   East  30.7           121.8        344.6   
547    P100548  73.0    Male   West  24.5           135.9        388.8   
815    P100816  53.0    Male  South  26.9           121.2        406.0   
955    P100956  84.0    Male   East  22.1           118.7        336.8   

     glucose  disease_risk  BMI_Missing  
78     105.0           0.0          0.0  
97     124.1           0.0          0.0  
118     97.1           0.0          0.0  
197     99.3     

### - IQR method:

In [21]:
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['bmi'] < lower_bound) | (df['bmi'] > upper_bound)]
print(outliers)

    patient_id   age  gender region   bmi  blood_pressure  cholesterol  \
45     P100046  64.0  Female  North  49.4           107.5        201.8   
286    P100287  43.0  Female  North  56.0           116.5        192.1   
320    P100321  70.0  Female   West  41.7           116.6        196.9   
345    P100346  51.0  Female   West  53.4           117.1        182.7   
360    P100361  79.0    Male  North  12.8           125.0        194.9   
423    P100424  47.0  Female   West  10.8           128.2        164.7   
445    P100446  70.0    Male  South  50.2           121.5        225.4   
574    P100575  62.0    Male  South  58.1           108.0        214.4   
616    P100617  23.0  Female   West  13.6           106.1        199.7   
697    P100698  43.0  Female  South  52.6           129.3        224.6   
811    P100812  32.0  Female   East  47.9           133.5        203.9   
909    P100910  39.0    Male   West  46.5           125.3        206.9   
931    P100932  36.0  Female   West  5

## - Percentile method:

In [24]:
lower_percentile = df['bmi'].quantile(0.01)
upper_percentile = df['bmi'].quantile(0.99)
df['bmi'] = df['bmi'].clip(lower=lower_percentile, upper=upper_percentile)
print(df['bmi'].describe())

count    1000.000000
mean       26.467454
std         4.679711
min        14.700000
25%        23.300000
50%        26.563854
75%        29.400000
max        41.700005
Name: bmi, dtype: float64


## 4. Apply Winsorization to cap extreme outliers instead of removing them.

In [26]:
from scipy.stats.mstats import winsorize
df['bmi'] = winsorize(df['bmi'], limits=[0.01, 0.01])
print(df['bmi'].describe())

count    1000.000000
mean       26.467454
std         4.679711
min        14.700000
25%        23.300000
50%        26.563854
75%        29.400000
max        41.700000
Name: bmi, dtype: float64


c:\Users\Divy\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


## 5. Compare dataset shape and summary before vs after outlier treatment.

In [28]:
print("Before outlier treatment:")
print(df.shape)
print(df['bmi'].describe())

print("\nAfter outlier treatment:")
print(df.shape)
print(df['bmi'].describe())

Before outlier treatment:
(1000, 10)
count    1000.000000
mean       26.467454
std         4.679711
min        14.700000
25%        23.300000
50%        26.563854
75%        29.400000
max        41.700000
Name: bmi, dtype: float64

After outlier treatment:
(1000, 10)
count    1000.000000
mean       26.467454
std         4.679711
min        14.700000
25%        23.300000
50%        26.563854
75%        29.400000
max        41.700000
Name: bmi, dtype: float64


c:\Users\Divy\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\Divy\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


# Part - C : Final Clean Dataset

## 6. Present the final cleaned dataset with: 

### - All missing values treated appropriately

In [30]:
df = df.dropna()
print("After treating missing values:")
print(df.isnull().sum())


After treating missing values:
patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
BMI_Missing       0
dtype: int64


### - All outliers handled using suitable methods.

In [33]:
numeric_cols = df.select_dtypes(include="number")
Q1 = numeric_cols.quantile(0.25)
Q3 = numeric_cols.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df[numeric_cols.columns] = numeric_cols.clip(lower=lower, upper=upper, axis=1)
print(df[numeric_cols.columns])

      age   bmi  blood_pressure  cholesterol  glucose  disease_risk  \
0    69.0  19.7           109.5   165.500000    103.0           0.0   
1    32.0  19.5           119.9   186.000000     78.3           0.0   
2    78.0  14.7           112.8   112.200000     91.9           0.0   
3    38.0  20.5            97.4   219.900000    114.9           0.0   
4    41.0  26.1           107.1   238.000000     81.1           0.0   
..    ...   ...             ...          ...      ...           ...   
995  61.0  27.0           131.9   171.200000     91.8           0.0   
996  41.0  25.1           125.4   195.100000     67.2           0.0   
997  47.0  28.8           108.2   256.800000    106.4           0.0   
998  76.0  31.0           115.0   153.100000     99.1           0.0   
999  31.0  22.3           121.8   195.606199     61.4           0.0   

     BMI_Missing  
0            0.0  
1            0.0  
2            0.0  
3            0.0  
4            0.0  
..           ...  
995          0